# Week 3, Lab 1 — Your first Crew

CrewAI = roles + tasks + a process. Point `LLM` at Ollama or the Colab compat server.


In [1]:
import zipfile
import os

zip_path = "/content/shared.zip"      # Path of the uploaded ZIP file
extract_path = "/content/shared"      # Folder where files will be extracted

# Create the folder if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ ZIP extracted successfully!")
print("Files extracted to:", extract_path)

✅ ZIP extracted successfully!
Files extracted to: /content/shared


In [2]:
import zipfile
import os

zip_path = "/content/shared.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully!")

✅ Extracted successfully!


In [3]:
WEEK = 'Week 3'
LAB = 'Lab 1 — first crew'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 3 / Lab 1 — first crew
Environment: Google Colab
Backend: huggingface
Tip: Runtime → Change runtime type → T4 GPU for faster generation.
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [5]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn crewai
else:
    %pip install -q crewai ollama


In [6]:
pip install crewai


In [8]:
!pip install -U litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 9.7 MB/s eta 0:00:00
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib_metadata 9.0.1
    Uninstalling importlib_metadata-9.0.1:
      Successfully uninstalled importlib_metadata-9.0.1


In [11]:
!pip uninstall -y crewai litellm
!pip install -U "crewai[litellm]"

Found existing installation: crewai 1.15.21
Uninstalling crewai-1.15.21:
  Successfully uninstalled crewai-1.15.21
Found existing installation: litellm 1.100.1
Uninstalling litellm-1.100.1:
  Successfully uninstalled litellm-1.100.1
  Using cached crewai-1.15.21-py3-none-any.whl.metadata (36 kB)
  Using cached litellm-1.100.1-cp310-abi3-manylinux_2_28_x86_64.whl.metadata (41 kB)
Using cached litellm-1.100.1-cp310-abi3-manylinux_2_28_x86_64.whl (24.1 MB)
Using cached crewai-1.15.21-py3-none-any.whl (1.1 MB)


In [11]:
from crewai import LLM, Agent, Task, Crew, Process

In [ ]:
from crewai import LLM

HF_TOKEN = "your HF token here"  # Replace with your actual Hugging Face token

llm = LLM(
    model="huggingface/meta-llama/Llama-3.1-8B-Instruct",
    api_key=HF_TOKEN,
    base_url="https://router.huggingface.co/v1",
    is_litellm=True        # Tell CrewAI to use LiteLLM
)

print("CrewAI is using Hugging Face!")

CrewAI is using Hugging Face!


In [16]:
from crewai import Agent, Task, Crew, Process

# ------------------ Agents ------------------

researcher = Agent(
    role="Researcher",
    goal="Find accurate information about the given topic using what you know and keep it factual.",
    backstory="A careful research analyst who prefers short bullet facts.",
    llm=llm,
    verbose=True,
)

writer = Agent(
    role="Writer",
    goal="Turn research notes into a 4-sentence student-friendly explanation.",
    backstory="A teacher who hates jargon.",
    llm=llm,
    verbose=True,
)

# ------------------ Tasks ------------------

t1 = Task(
    description="List 4 facts about MCP (Model Context Protocol).",
    expected_output="4 short bullet points about MCP.",
    agent=researcher,
)

t2 = Task(
    description="Write a beginner-friendly explanation of MCP in exactly 4 sentences using the research from the previous task.",
    expected_output="Exactly 4 simple sentences.",
    agent=writer,
)

# ------------------ Crew ------------------

crew = Crew(
    agents=[researcher, writer],
    tasks=[t1, t2],
    process=Process.sequential,
    verbose=True,
)

# ------------------ Run Crew (Colab/Jupyter) ------------------

result = await crew.kickoff_async()

print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 59c84e3c-f8de-45e2-be66-febf9e6216d7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: List 4 facts about MCP (Model Context Protocol).                                                         │
│  ID: 6b4b014b-b6f1-4f34-a4c7-da22fd6d94e1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Researcher                                                                                              │
│                                                                                                                 │
│  Task: List 4 facts about MCP (Model Context Protocol).                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Researcher                                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  • MCP (Model Context Protocol) is a network protocol used in IBM mainframe systems.                            │
│  • It is a variant of the SNA (Systems Network Architecture) protocol suite.                                    │
│  • MCP is used for network communication between systems on the same network.                                   │
│  • It is an older protocol, largely replaced by TCP/IP in modern systems.                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Writer                                                                                                  │
│                                                                                                                 │
│  Task: Write a beginner-friendly explanation of MCP in exactly 4 sentences using the research from the          │
│  previous task.                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Writer                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here is a beginner-friendly explanation of MCP in exactly 4 sentences:                                         │
│                                                                                                                 │
│  MCP, or Model Context Protocol, is a special network language that helps IBM mainframe systems talk to each    │
│  other on the same network. It's a part of a bigger set of rules called SNA (Systems Network Architecture)      │
│  that helps different systems understand each other. When two systems on the same network want to communicate,  │
│  they use MCP to send and receive messages. Although MCP is still used in some older systems, it's largely      │
│  been replaced by a newer protocol called TCP/IP in modern systems.                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a beginner-friendly explanation of MCP in exactly 4 sentences using the research from the          │
│  previous task.                                                                                                 │
│  Agent: Writer                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Here is a beginner-friendly explanation of MCP in exactly 4 sentences:

MCP, or Model Context Protocol, is a special network language that helps IBM mainframe systems talk to each other on the same network. It's a part of a bigger set of rules called SNA (Systems Network Architecture) that helps different systems understand each other. When two systems on the same network want to communicate, they use MCP to send and receive messages. Although MCP is still used in some older systems, it's largely been replaced by a newer protocol called TCP/IP in modern systems.


╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.      

Keep tasks tiny on small models. **Next:** backstory ablations.
